In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
N = 7200
TRAIN_END = 700
CAL_WINDOW = 300
LOCAL_WINDOW = 24
ALPHA = 0.10
GAMMA = 0.001
DECAY = 0.95

SHIFT_TIMES = [1800, 2800, 3800, 4800, 5800]
REGIME_SIGMAS = [0.06, 0.17, 0.09, 0.21, 0.10, 0.23]
UPWARD_SHIFTS = [1800, 3800, 5800]   
WINDOW_LEN = 180
LATE_OFFSET = 200                  

PHI = 0.8
INTERCEPT = 10.0
INITIAL_VALUE = 50.0
MIN_POSITIVE = 1e-6
SAVE_DIR = 'ar1_upward_shift_outputs_side_by_side'

# ============================================================
# DATA GENERATION
# ============================================================
def generate_series():
    rng = np.random.default_rng(SEED)
    boundaries = [0, *SHIFT_TIMES, N]
    sigma = np.empty(N, dtype=float)
    for j, (begin, end) in enumerate(zip(boundaries[:-1], boundaries[1:])):
        sigma[begin:end] = REGIME_SIGMAS[j]

    x = np.empty(N, dtype=float)
    x[0] = INITIAL_VALUE
    clipped = 0
    for t in range(1, N):
        noise = rng.normal(0.0, sigma[t])
        proposal = INTERCEPT + PHI * x[t - 1] + noise
        clipped += int(proposal <= MIN_POSITIVE)
        x[t] = max(proposal, MIN_POSITIVE)
    return x, sigma, clipped


def fit_ar1(x):
    design = np.column_stack([np.ones(TRAIN_END - 1), x[:TRAIN_END - 1]])
    target = x[1:TRAIN_END]
    intercept_hat, phi_hat = np.linalg.lstsq(design, target, rcond=None)[0]
    return float(intercept_hat), float(phi_hat)


def conformal_radius(scores, alpha_level):
    if alpha_level <= 0:
        return np.inf
    if alpha_level >= 1:
        return -np.inf
    n = len(scores)
    if n == 0:
        raise ValueError('At least one residual score is required')
    k = int(np.ceil((n + 1) * (1 - alpha_level)))
    if k > n:
        return np.inf
    if k < 1:
        return -np.inf
    return float(np.partition(scores, k - 1)[k - 1])


def make_bounds(prediction, radius):
    if np.isposinf(radius):
        return -np.inf, np.inf
    if np.isneginf(radius):
        return np.inf, -np.inf  # empty interval
    return float(prediction - radius), float(prediction + radius)



def run_aci(x, model):
    intercept_hat, phi_hat = model
    training_predictions = intercept_hat + phi_hat * x[:TRAIN_END - 1]
    prior_scores = np.abs(x[1:TRAIN_END] - training_predictions).tolist()

    alpha_t = ALPHA
    weighted_num = 0.0
    weighted_den = 0.0
    coverage_history = []
    rows = []

    for t in range(TRAIN_END, N):
        scores = np.asarray(prior_scores[-CAL_WINDOW:], dtype=float)
        radius = conformal_radius(scores, alpha_t)
        pred = intercept_hat + phi_hat * x[t - 1]
        lower, upper = make_bounds(pred, radius)
        err = int(not (lower <= x[t] <= upper))

        coverage_history.append(1 - err)
        if len(coverage_history) >= LOCAL_WINDOW:
            lpicp = float(np.mean(coverage_history[-LOCAL_WINDOW:]))
            le = max((1 - ALPHA) - lpicp, 0.0)
        else:
            lpicp = np.nan
            le = np.nan

        # Recency-weighted ACI update
        weighted_num = DECAY * weighted_num + err
        weighted_den = DECAY * weighted_den + 1.0
        weighted_err = weighted_num / weighted_den

        alpha_used = alpha_t
        alpha_t = alpha_t + GAMMA * (ALPHA - weighted_err)

        rows.append((t, x[t], pred, lower, upper, err, lpicp, le,
                     alpha_used, alpha_t, weighted_err))

        prior_scores.append(abs(x[t] - pred))

    cols = ['t', 'y', 'point_forecast', 'lower', 'upper', 'err',
            'LPICP', 'LE', 'alpha_used', 'alpha_next', 'weighted_error']
    return pd.DataFrame(rows, columns=cols).set_index('t')


def running_le_baseline(le_series):
    out = np.full(len(le_series), np.nan)
    numerator = 0.0
    denominator = 0.0
    values = le_series.to_numpy(dtype=float)
    for i, v in enumerate(values):
        if np.isfinite(v):
            numerator = DECAY * numerator + v
            denominator = DECAY * denominator + 1.0
            out[i] = numerator / denominator
    return pd.Series(out, index=le_series.index)


# ============================================================
# WINDOW SUMMARIES
# ============================================================
def summarize_window(frame, start, length, label, sigma_before, sigma_after):
    end = start + length - 1
    sub = frame.loc[start:end].copy()
    valid_failures = sub[(sub['err'] == 1) & sub['LE'].notna() & sub['LEbar'].notna()].copy()
    n_fail = int(len(valid_failures))
    n_above = int((valid_failures['LE'] > valid_failures['LEbar']).sum())
    pct_above = 100.0 * n_above / n_fail if n_fail > 0 else np.nan
    return {
        'window_label': label,
        'window_start': int(start),
        'window_end': int(end),
        'sigma_before': float(sigma_before),
        'sigma_after': float(sigma_after),
        'n_points': int(len(sub)),
        'n_failures': n_fail,
        'n_LE_above_LEbar': n_above,
        'pct_LE_above_LEbar': pct_above,
        'mean_LE_failures': float(valid_failures['LE'].mean()) if n_fail else np.nan,
        'mean_LEbar_failures': float(valid_failures['LEbar'].mean()) if n_fail else np.nan,
        'mean_gap': float((valid_failures['LE'] - valid_failures['LEbar']).mean()) if n_fail else np.nan,
    }


def plot_two_windows_side_by_side(frame, shift, save_dir):
    sigma_idx = SHIFT_TIMES.index(shift)
    sigma_before = REGIME_SIGMAS[sigma_idx]
    sigma_after = REGIME_SIGMAS[sigma_idx + 1]

    immediate_start = shift
    late_start = shift + LATE_OFFSET
    immediate = summarize_window(frame, immediate_start, WINDOW_LEN, 'immediate', sigma_before, sigma_after)
    late = summarize_window(frame, late_start, WINDOW_LEN, 'late', sigma_before, sigma_after)

    windows = [
        ('Immediate after shift', immediate_start, immediate),
        (f'{LATE_OFFSET} points after shift', late_start, late),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)

    # Common y-range from both windows
    all_vals = []
    for _, start, _ in windows:
        sub = frame.loc[start:start + WINDOW_LEN - 1]
        vals = np.concatenate([
            sub['LE'].to_numpy(dtype=float),
            sub['LEbar'].to_numpy(dtype=float)
        ])
        vals = vals[np.isfinite(vals)]
        if len(vals):
            all_vals.append(vals)
    if all_vals:
        vals = np.concatenate(all_vals)
        ymin, ymax = np.min(vals), np.max(vals)
        pad = 0.05 * (ymax - ymin) if ymax > ymin else 0.02
        ylims = (max(0.0, ymin - pad), ymax + pad)
    else:
        ylims = (0.0, 1.0)

    for ax, (title, start, stats) in zip(axes, windows):
        end = start + WINDOW_LEN - 1
        sub = frame.loc[start:end].copy()
        tt = sub.index.to_numpy()
        le = sub['LE'].to_numpy(dtype=float)
        lebar = sub['LEbar'].to_numpy(dtype=float)
        err = sub['err'].to_numpy(dtype=int)

        ax.plot(tt, le, color='#dc2626', lw=2.1, label=r'$LE_s^{(L)}$')
        ax.plot(tt, lebar, color='#2563eb', lw=2.1, ls='--', label=r' $\,\overline{LE}_s^{(L)}$')

        valid_fail = (err == 1) & np.isfinite(le) & np.isfinite(lebar)
        above = valid_fail & (le > lebar)
        below = valid_fail & (le <= lebar)

        ax.scatter(tt[above], le[above], marker='x', s=70, linewidths=2,
                   color='#b91c1c', label=r'Failure with $LE_s^{(L)} > \overline{LE}_s^{(L)}$', zorder=5)
        ax.scatter(tt[below], le[below], marker='o', s=40,
                   color='#1d4ed8', edgecolors='white', linewidths=0.6,
                   label=r'Failure with $LE_s^{(L)} \leq \overline{LE}_s^{(L)}$', zorder=5)

        if title.startswith('Immediate'):
            ax.axvline(shift, color='black', ls=':', lw=1.2, label=f'Shift at t={shift}')
        else:
            ax.axvline(shift, color='black', ls=':', lw=1.2)

        pct_text = 'NA' if not np.isfinite(stats['pct_LE_above_LEbar']) else f"{stats['pct_LE_above_LEbar']:.2f}%"
        ax.set_title(
            f"{title}\n"
            f"Window: [{start}, {end}]\n"
            f"Failures above LE_above_LEbar: {stats['n_LE_above_LEbar']}/{stats['n_failures']} ({pct_text})",
            fontsize=10.5
        )
        ax.set_xlabel('Time index')
        ax.grid(alpha=0.25)
        ax.set_ylim(*ylims)

    axes[0].set_ylabel(r'Local undercoverage error')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.03))
    fig.suptitle(
        f'ACI local error around upward variance shift at t={shift} '
        f'($\\sigma$: {sigma_before:.2f} $\\to$ {sigma_after:.2f})',
        y=1.08, fontsize=13
    )
    fig.tight_layout()

    out_path = save_dir / f'shift_{shift}_side_by_side.png'
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    return immediate, late, out_path


# ============================================================
# MAIN
# ============================================================
def main():
    output_dir = Path(SAVE_DIR)   # fixes mkdir problem when SAVE_DIR is a string
    output_dir.mkdir(parents=True, exist_ok=True)

    x, sigma, clipped = generate_series()
    model = fit_ar1(x)
    aci = run_aci(x, model)
    aci['LEbar'] = running_le_baseline(aci['LE'])

    # sanity check
    observed_err = ((aci['y'] < aci['lower']) | (aci['y'] > aci['upper'])).astype(int)
    if not np.array_equal(observed_err.to_numpy(), aci['err'].to_numpy()):
        raise ValueError('Stored errors do not match interval coverage.')

    aci.to_csv(output_dir / 'aci_trajectory.csv')

    all_rows = []
    first180_rows = []
    image_paths = []

    for shift in UPWARD_SHIFTS:
        immediate_start = shift
        late_start = shift + LATE_OFFSET
        late_end = late_start + WINDOW_LEN - 1

        next_change = next((s for s in SHIFT_TIMES if s > shift), N)
        if late_end >= next_change:
            raise ValueError(
                f'Late window for shift {shift} crosses next shift/end. '
                f'late_end={late_end}, next_change={next_change}'
            )

        immediate, late, img_path = plot_two_windows_side_by_side(aci, shift, output_dir)
        image_paths.append(img_path)

        for row in (immediate, late):
            row2 = {'shift_time': shift}
            row2.update(row)
            all_rows.append(row2)

        first180_row = {
            'shift_time': shift,
            'sigma_before': immediate['sigma_before'],
            'sigma_after': immediate['sigma_after'],
            'window_start': immediate['window_start'],
            'window_end': immediate['window_end'],
            'n_failures': immediate['n_failures'],
            'n_LE_above_LEbar': immediate['n_LE_above_LEbar'],
            'fraction': (f"{immediate['n_LE_above_LEbar']}/{immediate['n_failures']}"
                         if immediate['n_failures'] > 0 else 'NA'),
            'pct_LE_above_LEbar': immediate['pct_LE_above_LEbar'],
            'mean_LE_failures': immediate['mean_LE_failures'],
            'mean_LEbar_failures': immediate['mean_LEbar_failures'],
            'mean_gap': immediate['mean_gap'],
        }
        first180_rows.append(first180_row)

    all_df = pd.DataFrame(all_rows)
    first180_df = pd.DataFrame(first180_rows)

    all_df.to_csv(output_dir / 'upward_shift_failure_fractions.csv', index=False)
    first180_df.to_csv(output_dir / 'upward_shifts_first_180.csv', index=False)

    print('=== Upward variance shifts only ===')
    print(first180_df.to_string(index=False, float_format=lambda z: f'{z:.5f}'))
    print(f'\nOLS fit: intercept={model[0]:.6f}, phi={model[1]:.6f}')
    print(f'Positive clipping events: {clipped}')
    print(f'Saved CSVs and {len(image_paths)} side-by-side images to: {output_dir.resolve()}')
    for p in image_paths:
        print(' -', p)


if __name__ == '__main__':
    main()


=== Upward variance shifts only ===
 shift_time  sigma_before  sigma_after  window_start  window_end  n_failures  n_LE_above_LEbar fraction  pct_LE_above_LEbar  mean_LE_failures  mean_LEbar_failures  mean_gap
       1800       0.06000      0.17000          1800        1979          54                36    36/54            66.66667           0.28025              0.20534   0.07491
       3800       0.09000      0.21000          3800        3979          52                40    40/52            76.92308           0.27965              0.23246   0.04719
       5800       0.10000      0.23000          5800        5979          50                37    37/50            74.00000           0.22500              0.18238   0.04262

OLS fit: intercept=10.366624, phi=0.792620
Positive clipping events: 0
Saved CSVs and 3 side-by-side images to: C:\Users\DAIICT J\Downloads\TLMR_Local_Gitub\05.Synthetic_Exp\ar1_upward_shift_outputs_side_by_side
 - ar1_upward_shift_outputs_side_by_side\shift_1800_side_by

In [ ]:
from pathlib import Path

import numpy as np

import pandas as pd

import matplotlib

matplotlib.use('Agg')

import matplotlib.pyplot as plt




SEED = 42

N = 7200

TRAIN_END = 700

CAL_WINDOW = 300

LOCAL_WINDOW = 24

ALPHA = 0.10

GAMMA = 0.001

DECAY = 0.95



SHIFT_TIMES = [1800, 2800, 3800, 4800, 5800]

REGIME_SIGMAS = [0.06, 0.17, 0.09, 0.21, 0.10, 0.23]

UPWARD_SHIFTS = [1800, 3800, 5800]   # keep only variance-increase shifts

WINDOW_LEN = 180

LATE_OFFSET = 200                   



PHI = 0.8

INTERCEPT = 10.0

INITIAL_VALUE = 50.0

MIN_POSITIVE = 1e-6

SAVE_DIR = './ar1_all_upward_shifts_side_by_side'



# ============================================================

# DATA GENERATION

# ============================================================

def generate_series():

    rng = np.random.default_rng(SEED)

    boundaries = [0, *SHIFT_TIMES, N]

    sigma = np.empty(N, dtype=float)

    for j, (begin, end) in enumerate(zip(boundaries[:-1], boundaries[1:])):

        sigma[begin:end] = REGIME_SIGMAS[j]



    x = np.empty(N, dtype=float)

    x[0] = INITIAL_VALUE

    clipped = 0

    for t in range(1, N):

        noise = rng.normal(0.0, sigma[t])

        proposal = INTERCEPT + PHI * x[t - 1] + noise

        clipped += int(proposal <= MIN_POSITIVE)

        x[t] = max(proposal, MIN_POSITIVE)

    return x, sigma, clipped





def fit_ar1(x):

    design = np.column_stack([np.ones(TRAIN_END - 1), x[:TRAIN_END - 1]])

    target = x[1:TRAIN_END]

    intercept_hat, phi_hat = np.linalg.lstsq(design, target, rcond=None)[0]

    return float(intercept_hat), float(phi_hat)





def conformal_radius(scores, alpha_level):

    if alpha_level <= 0:

        return np.inf

    if alpha_level >= 1:

        return -np.inf

    n = len(scores)

    if n == 0:

        raise ValueError('At least one residual score is required')

    k = int(np.ceil((n + 1) * (1 - alpha_level)))

    if k > n:

        return np.inf

    if k < 1:

        return -np.inf

    return float(np.partition(scores, k - 1)[k - 1])





def make_bounds(prediction, radius):

    if np.isposinf(radius):

        return -np.inf, np.inf

    if np.isneginf(radius):

        return np.inf, -np.inf  # empty interval

    return float(prediction - radius), float(prediction + radius)





# ============================================================

# ACI ONLY 

# ============================================================

def run_aci(x, model):

    intercept_hat, phi_hat = model

    training_predictions = intercept_hat + phi_hat * x[:TRAIN_END - 1]

    prior_scores = np.abs(x[1:TRAIN_END] - training_predictions).tolist()



    alpha_t = ALPHA

    weighted_num = 0.0

    weighted_den = 0.0

    coverage_history = []

    rows = []



    for t in range(TRAIN_END, N):

        scores = np.asarray(prior_scores[-CAL_WINDOW:], dtype=float)

        radius = conformal_radius(scores, alpha_t)

        pred = intercept_hat + phi_hat * x[t - 1]

        lower, upper = make_bounds(pred, radius)

        err = int(not (lower <= x[t] <= upper))



        coverage_history.append(1 - err)

        if len(coverage_history) >= LOCAL_WINDOW:

            lpicp = float(np.mean(coverage_history[-LOCAL_WINDOW:]))

            le = max((1 - ALPHA) - lpicp, 0.0)

        else:

            lpicp = np.nan

            le = np.nan



        # Recency-weighted ACI update

        weighted_num = DECAY * weighted_num + err

        weighted_den = DECAY * weighted_den + 1.0

        weighted_err = weighted_num / weighted_den



        alpha_used = alpha_t

        alpha_t = alpha_t + GAMMA * (ALPHA - weighted_err)



        rows.append((t, x[t], pred, lower, upper, err, lpicp, le,

                     alpha_used, alpha_t, weighted_err))



        prior_scores.append(abs(x[t] - pred))



    cols = ['t', 'y', 'point_forecast', 'lower', 'upper', 'err',

            'LPICP', 'LE', 'alpha_used', 'alpha_next', 'weighted_error']

    return pd.DataFrame(rows, columns=cols).set_index('t')





def running_le_baseline(le_series):

    out = np.full(len(le_series), np.nan)

    numerator = 0.0

    denominator = 0.0

    values = le_series.to_numpy(dtype=float)

    for i, v in enumerate(values):

        if np.isfinite(v):

            numerator = DECAY * numerator + v

            denominator = DECAY * denominator + 1.0

            out[i] = numerator / denominator

    return pd.Series(out, index=le_series.index)





# ============================================================

# WINDOW SUMMARIES

# ============================================================

def summarize_window(frame, start, length, label, sigma_before, sigma_after):

    end = start + length - 1

    sub = frame.loc[start:end].copy()

    valid_failures = sub[(sub['err'] == 1) & sub['LE'].notna() & sub['LEbar'].notna()].copy()

    n_fail = int(len(valid_failures))

    n_above = int((valid_failures['LE'] > valid_failures['LEbar']).sum())

    pct_above = 100.0 * n_above / n_fail if n_fail > 0 else np.nan

    return {

        'window_label': label,

        'window_start': int(start),

        'window_end': int(end),

        'sigma_before': float(sigma_before),

        'sigma_after': float(sigma_after),

        'n_points': int(len(sub)),

        'n_failures': n_fail,

        'n_LE_above_LEbar': n_above,

        'pct_LE_above_LEbar': pct_above,

        'mean_LE_failures': float(valid_failures['LE'].mean()) if n_fail else np.nan,

        'mean_LEbar_failures': float(valid_failures['LEbar'].mean()) if n_fail else np.nan,

        'mean_gap': float((valid_failures['LE'] - valid_failures['LEbar']).mean()) if n_fail else np.nan,

    }





def plot_two_windows_side_by_side(frame, shift, save_dir):
    """Reference-style pair of plots, retaining the original window statistics."""
    sigma_idx = SHIFT_TIMES.index(shift)
    sigma_before = REGIME_SIGMAS[sigma_idx]
    sigma_after = REGIME_SIGMAS[sigma_idx + 1]

    immediate_start = shift
    late_start = shift + LATE_OFFSET
    immediate = summarize_window(frame, immediate_start, WINDOW_LEN, 'immediate', sigma_before, sigma_after)
    late = summarize_window(frame, late_start, WINDOW_LEN, 'late', sigma_before, sigma_after)
    windows = [
        ('Immediately after shift', immediate_start, immediate),
        (f'{LATE_OFFSET} steps after shift', late_start, late),
    ]

  
    fig, axes = plt.subplots(1, 2, figsize=(19, 5.2), sharey=False)
    for ax, (title, start, stats) in zip(axes, windows):
        end = start + WINDOW_LEN - 1
        sub = frame.loc[start:end]
        tt = sub.index.to_numpy()
        le = sub['LE'].to_numpy(dtype=float)
        lebar = sub['LEbar'].to_numpy(dtype=float)
        err = sub['err'].to_numpy(dtype=int)

        ax.plot(tt, le, color='#c02d20', lw=2.2, label=r'$LE_s^{(L)}$')
        ax.plot(tt, lebar, color='#2563eb', lw=2.2,
                label=r'$\overline{LE}_s^{(L)}$')

        valid_fail = (err == 1) & np.isfinite(le) & np.isfinite(lebar)
        above = valid_fail & (le > lebar)
        below = valid_fail & (le <= lebar)
        ax.scatter(tt[above], le[above], marker='x', s=70,
                   linewidths=2.0, color='#c02d20',
                   label='Failure: LE > $\overline{LE}_s^{(L)}$', zorder=5)
        ax.scatter(tt[below], le[below], marker='o', s=40,
                   facecolors='white', edgecolors='#3b82f6', linewidths=1.5,
                   label='Failure: LE <= $\overline{LE}_s^{(L)}$', zorder=5)

        ax.set_title(f'Shift t={shift}: {title} [{start}, {end}]', fontsize=14)
        ax.set_xlabel('Time index', fontsize=12)
        ax.set_ylabel('Left centered Local error ($LE_s^{(L)}$)', fontsize=12)
        ax.grid(alpha=0.23)
        ax.legend(loc='upper left', fontsize=10, frameon=True)
        ax.set_xlim(start - 9, end + 9)
        finite = np.concatenate([le[np.isfinite(le)], lebar[np.isfinite(lebar)]])
        top = float(np.max(finite)) if len(finite) else 1.0
        ax.set_ylim(0, max(0.01, top * 1.13))

        n_above = stats['n_LE_above_LEbar']
        n_fail = stats['n_failures']
        pct_fail = 100.0 * n_above / n_fail if n_fail else float('nan')
        n_obs = stats['n_points']
        pct_obs = 100.0 * n_above / n_obs if n_obs else float('nan')
        label = (f'{n_above}/{n_fail} failures = {pct_fail:.2f}%\n'
                 f'{n_above}/{n_obs} observations = {pct_obs:.2f}%')
        ax.text(.988, .98, label, transform=ax.transAxes, ha='right', va='top',
                fontsize=10, bbox=dict(facecolor='white', edgecolor='black',
                                       boxstyle='round,pad=0.25', linewidth=1.25))

    fig.tight_layout(w_pad=3)
    out_path = save_dir / f'shift_{shift}_side_by_side.png'
    fig.savefig(out_path, dpi=250, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return immediate, late, out_path


def main():

    output_dir = Path(SAVE_DIR)  

    output_dir.mkdir(parents=True, exist_ok=True)



    x, sigma, clipped = generate_series()

    model = fit_ar1(x)

    aci = run_aci(x, model)

    aci['LEbar'] = running_le_baseline(aci['LE'])



    # sanity check

    observed_err = ((aci['y'] < aci['lower']) | (aci['y'] > aci['upper'])).astype(int)

    if not np.array_equal(observed_err.to_numpy(), aci['err'].to_numpy()):

        raise ValueError('Stored errors do not match interval coverage.')



    aci.to_csv(output_dir / 'aci_trajectory.csv')



    all_rows = []

    first180_rows = []

    image_paths = []



    for shift in UPWARD_SHIFTS:

        immediate_start = shift

        late_start = shift + LATE_OFFSET

        late_end = late_start + WINDOW_LEN - 1



        next_change = next((s for s in SHIFT_TIMES if s > shift), N)

        if late_end >= next_change:

            raise ValueError(

                f'Late window for shift {shift} crosses next shift/end. '

                f'late_end={late_end}, next_change={next_change}'

            )



        immediate, late, img_path = plot_two_windows_side_by_side(aci, shift, output_dir)
        image_paths.append(img_path)

        for row in (immediate, late):

            row2 = {'shift_time': shift}

            row2.update(row)

            all_rows.append(row2)



        first180_row = {

            'shift_time': shift,

            'sigma_before': immediate['sigma_before'],

            'sigma_after': immediate['sigma_after'],

            'window_start': immediate['window_start'],

            'window_end': immediate['window_end'],

            'n_failures': immediate['n_failures'],

            'n_LE_above_LEbar': immediate['n_LE_above_LEbar'],

            'fraction': (f"{immediate['n_LE_above_LEbar']}/{immediate['n_failures']}"

                         if immediate['n_failures'] > 0 else 'NA'),

            'pct_LE_above_LEbar': immediate['pct_LE_above_LEbar'],

            'mean_LE_failures': immediate['mean_LE_failures'],

            'mean_LEbar_failures': immediate['mean_LEbar_failures'],

            'mean_gap': immediate['mean_gap'],

        }

        first180_rows.append(first180_row)



    all_df = pd.DataFrame(all_rows)

    first180_df = pd.DataFrame(first180_rows)



    all_df.to_csv(output_dir / 'upward_shift_failure_fractions.csv', index=False)

    first180_df.to_csv(output_dir / 'upward_shifts_first_180.csv', index=False)



    print('=== Upward variance shifts only ===')

    print(first180_df.to_string(index=False, float_format=lambda z: f'{z:.5f}'))

    print(f'\nOLS fit: intercept={model[0]:.6f}, phi={model[1]:.6f}')

    print(f'Positive clipping events: {clipped}')

    print(f'Saved CSVs and {len(image_paths)} side-by-side images to: {output_dir.resolve()}')

    for p in image_paths:

        print(' -', p)





if __name__ == '__main__':

    main()


=== Upward variance shifts only ===
 shift_time  sigma_before  sigma_after  window_start  window_end  n_failures  n_LE_above_LEbar fraction  pct_LE_above_LEbar  mean_LE_failures  mean_LEbar_failures  mean_gap
       1800       0.06000      0.17000          1800        1979          54                36    36/54            66.66667           0.28025              0.20534   0.07491
       3800       0.09000      0.21000          3800        3979          52                40    40/52            76.92308           0.27965              0.23246   0.04719
       5800       0.10000      0.23000          5800        5979          50                37    37/50            74.00000           0.22500              0.18238   0.04262

OLS fit: intercept=10.366624, phi=0.792620
Positive clipping events: 0
Saved CSVs and 3 side-by-side images to: C:\Users\DAIICT J\Downloads\TLMR_Local_Gitub\05.Synthetic_Exp\ar1_all_upward_shifts_side_by_side
 - ar1_all_upward_shifts_side_by_side\shift_1800_side_by_side.